In [ ]:
!pip install ultralytics
!pip install torch
from ultralytics import YOLO
import torch

!rm -f /content/drive/MyDrive/Sprocket3BS/yolo_data/train/labels.cache
!rm -f /content/drive/MyDrive/Sprocket3BS/yolo_data/val/labels.cache
!rm -f /content/drive/MyDrive/Sprocket3BS/yolo_data/test/labels.cache

print("--- HARDWARE CHECK ---")
if torch.cuda.is_available():
    print(f"Success. Found GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found. The script will run on your CPU.")
print("----------------------\n")


model = YOLO('yolo12l.pt')


# model.train(
#     data='/content/drive/MyDrive/Sprocket3BS/data.yaml',
#     epochs=250,
#     imgsz=1088,          # Use the full resolution of your tall images
#     rect=True,           # HUGE for 640x1088; saves VRAM and compute
#     batch=32,            # Increase to 48 if you have VRAM left
#     device=0,
#     workers=8,
#     amp=True,            # Ensure Automatic Mixed Precision is on (default is True)
#     project='/content/drive/MyDrive/Sprocket3BS',
#     name='version_12',
#     exist_ok=True,
#     patience=40,

#     # --- AUGMENTATION ---
#     conf=0.15,
#     mosaic=0.8,
#     mixup=0.15,           # Added: Good for robots overlapping each other
#     copy_paste=0.1,      # Added: Helps if you have few game piece samples
#     hsv_h=0.015,
#     hsv_s=0.5,
#     hsv_v=0.3,
#     degrees=10.0,
#     fliplr=0.5,
#     label_smoothing=0.05
# )

#fine tune
checkpoint_path = '//content/drive/MyDrive/Sprocket3BS/version_12/weights/best.pt'
model = YOLO(checkpoint_path)
model.train(
    data='/content/drive/MyDrive/Sprocket3BS/data.yaml',
    imgsz=1088,          # Use the full resolution of your tall images
    rect=True,           # HUGE for 640x1088; saves VRAM and compute
    batch=32,            # Increase to 48 if you have VRAM left
    device=0,
    workers=8,
    amp=True,            # Ensure Automatic Mixed Precision is on (default is True)
    project='/content/drive/MyDrive/Sprocket3BS',
    name='version_12.1',
    exist_ok=True,
    epochs=20,
    lr0=0.0001,      # 10x smaller than your main run
    patience=0,       # Force it to finish the "polish"

    # Kill the "Fake" image generators
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,

    # Keep the "Real-world" lighting/color variations
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.2,
    fliplr=0.5,

    # This is the "secret sauce" for Precision
    label_smoothing=0.1  # Slightly higher to prevent overfitting on specific pixels
)

print("\n--- TRAINING COMPLETE ---")

--- HARDWARE CHECK ---
Success. Found GPU: NVIDIA H100 80GB HBM3
----------------------

WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81079MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Sprocket3BS/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.2, imgsz=1088, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300,

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/Sprocket3BS/version_5_frc/weights/best.pt')

results = model.val(
    data='/content/drive/MyDrive/Sprocket3BS/data.yaml',
    split='test',
    project='/content/drive/MyDrive/Sprocket3BS',
    name='2025',
    exist_ok=True
)

Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,413,574 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 1.2±1.6 ms, read: 51.5±70.1 MB/s, size: 291.8 KB)
val: Scanning /content/drive/MyDrive/Sprocket3BS/yolo_data/test/labels... 0 images, 160 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 160/160 107.8it/s 1.5s
WARNING ⚠️ val: No labels found in /content/drive/MyDrive/Sprocket3BS/yolo_data/test/labels.cache. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
val: New cache created: /content/drive/MyDrive/Sprocket3BS/yolo_data/test/labels.cache
WARNING ⚠️ Labels are missing or empty in /content/drive/MyDrive/Sprocket3BS/yolo_data/test/labels.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.5it/s 6

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:657: RuntimeWarning: Mean of empty slice.
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:703: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:703: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.12/dist-packages/ultraly

                   all        160          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels
Speed: 12.9ms preprocess, 14.7ms inference, 0.0ms loss, 4.2ms postprocess per image
Results saved to /content/drive/MyDrive/Sprocket3BS/2025


In [ ]:
from ultralytics import YOLO

# Load your model
model = YOLO('/content/drive/MyDrive/Sprocket3BS/version_5_frc/weights/best.pt')

# Run Prediction (Inference) on your new unlabeled data
results = model.predict(
    source='/content/drive/MyDrive/Sprocket3BS/yolo_data/test/images', # Path to your folder of images
    conf=0.25,           # Confidence threshold (adjust as needed)
    save=True,           # Save the images with bounding box overlays
    save_txt=True,       # Save the predictions as .txt files (YOLO format labels)
    project='/content/drive/MyDrive/Sprocket3BS',
    name='2025_predictions',
    exist_ok=True
)


image 1/160 /content/drive/MyDrive/Sprocket3BS/yolo_data/test/images/Xqqco54CAdI_f0000000.jpg: 640x1088 2 Red Alliance Robots, 3 Blue Alliance Robots, 57.8ms
image 2/160 /content/drive/MyDrive/Sprocket3BS/yolo_data/test/images/Xqqco54CAdI_f0000030.jpg: 640x1088 4 Red Alliance Robots, 2 Blue Alliance Robots, 27.5ms
image 3/160 /content/drive/MyDrive/Sprocket3BS/yolo_data/test/images/Xqqco54CAdI_f0000060.jpg: 640x1088 4 Red Alliance Robots, 25.1ms
image 4/160 /content/drive/MyDrive/Sprocket3BS/yolo_data/test/images/Xqqco54CAdI_f0000090.jpg: 640x1088 5 Red Alliance Robots, 3 Blue Alliance Robots, 25.1ms
image 5/160 /content/drive/MyDrive/Sprocket3BS/yolo_data/test/images/Xqqco54CAdI_f0000120.jpg: 640x1088 4 Red Alliance Robots, 2 Blue Alliance Robots, 25.1ms
image 6/160 /content/drive/MyDrive/Sprocket3BS/yolo_data/test/images/Xqqco54CAdI_f0000150.jpg: 640x1088 5 Red Alliance Robots, 1 Blue Alliance Robot, 25.1ms
image 7/160 /content/drive/MyDrive/Sprocket3BS/yolo_data/test/images/Xqqco54

In [ ]:
import yaml

tracker_config = {
    'tracker_type': 'botsort',

    # Detection thresholds
    'track_high_thresh': 0.5,
    'track_low_thresh': 0.1,
    'new_track_thresh': 0.45,

    # Track lifecycle
    'track_buffer': 120,
    'match_thresh': 0.7,

    # Score fusion
    'fuse_score': True,

    # Global Motion Compensation (handles broadcast camera pan/zoom)
    'gmc_method': 'orb',

    # ReID (appearance-based matching for occlusions/collisions)
    'with_reid': True,
    'proximity_thresh': 0.5,
    'appearance_thresh': 0.25,
    'model': 'auto',
}

with open('custom_tracker.yaml', 'w') as f:
    yaml.dump(tracker_config, f)
!pip install -U -q ultralytics yt-dlp
import os
import shutil
from ultralytics import YOLO

# --- SETTINGS ---
YOUTUBE_URL = 'https://www.youtube.com/watch?v=oCgvl2cHzek'
START_TIME = '00:00:05'
END_TIME   = '00:02:50'

SAVE_PATH = '/content/drive/MyDrive/Sprocket3BS/version_12.1'
MODEL_PATH = os.path.join(f'{SAVE_PATH}/weights/best.pt')
RAW_VIDEO = '/content/raw_video.mp4'
FIXED_VIDEO = '/content/fixed_video.mp4'
LOCAL_DIR = '/content/process_test'
COMPRESSED_VIDEO = '/content/compressed_output.mp4'

DOWNLOAD_VIDEO = True   # True = download fresh from YouTube, False = reuse existing FIXED_VIDEO

if DOWNLOAD_VIDEO:
    # 1. Cleanup
    for f in [RAW_VIDEO, FIXED_VIDEO, COMPRESSED_VIDEO]:
        if os.path.exists(f): os.remove(f)
    if os.path.exists(LOCAL_DIR): shutil.rmtree(LOCAL_DIR)

    # Helper: convert HH:MM:SS or 'SS' to seconds for yt-dlp's download-sections
    def to_seconds(t):
        if t is None:
            return None
        parts = str(t).split(':')
        parts = [float(p) for p in parts]
        if len(parts) == 3:
            return parts[0] * 3600 + parts[1] * 60 + parts[2]
        if len(parts) == 2:
            return parts[0] * 60 + parts[1]
        return parts[0]

    start_s = to_seconds(START_TIME)
    end_s = to_seconds(END_TIME)
    section = f"*{start_s}-{end_s if end_s is not None else 'inf'}"

    # 2. Download + trim in one step via yt-dlp
    print(f"--- Downloading via yt-dlp (section {section}) ---")
    !yt-dlp --force-overwrites \
        --download-sections "{section}" \
        --force-keyframes-at-cuts \
        -f "bestvideo[ext=mp4][height<=720]+bestaudio[ext=m4a]/best[ext=mp4][height<=720]/best" \
        --merge-output-format mp4 \
        "{YOUTUBE_URL}" -o "{RAW_VIDEO}"

    # 3. Re-encode for OpenCV compatibility
    print("--- Converting Codec ---")
    !ffmpeg -i {RAW_VIDEO} -c:v libx264 -preset ultrafast -crf 23 -c:a copy {FIXED_VIDEO} -y -loglevel quiet
else:
    # Reusing existing video — just clean up stale inference artifacts
    print(f"--- Skipping download, reusing {FIXED_VIDEO} ---")
    if not os.path.exists(FIXED_VIDEO):
        raise FileNotFoundError(
            f"DOWNLOAD_VIDEO is False but {FIXED_VIDEO} doesn't exist. "
            f"Set DOWNLOAD_VIDEO = True to download it first."
        )
    if os.path.exists(COMPRESSED_VIDEO): os.remove(COMPRESSED_VIDEO)
    if os.path.exists(LOCAL_DIR): shutil.rmtree(LOCAL_DIR)

# 4. Final Verification and Inference
if os.path.exists(FIXED_VIDEO) and os.path.getsize(FIXED_VIDEO) > 1000:
    print(f"--- Processing Video ({os.path.getsize(FIXED_VIDEO)/1e6:.2f} MB) ---")
    model = YOLO(MODEL_PATH)
    results = model.track(
        batch=8,
        source=FIXED_VIDEO,
        project=LOCAL_DIR,
        name='output',
        save=True,
        stream=True,
        conf=0.25,
        device=0,
        tracker='custom_tracker.yaml'
    )

    frame_count = 0
    for r in results:
        frame_count += 1
        if frame_count % 100 == 0:
            print(f"Frame {frame_count} processed...")

    if frame_count > 0:
        search_dir = os.path.join(LOCAL_DIR, 'output')
        files = os.listdir(search_dir) if os.path.exists(search_dir) else []
        if files:
            result_file = os.path.join(search_dir, files[0])
            final_dest = os.path.join(SAVE_PATH, "frc_detections_v12.1.mp4")

            # Compress before saving to Drive
            print("--- Compressing output video ---")
            !ffmpeg -i {result_file} \
                -c:v libx264 \
                -preset medium \
                -crf 28 \
                -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2" \
                -c:a aac -b:a 128k \
                {COMPRESSED_VIDEO} -y -stats

            if os.path.exists(COMPRESSED_VIDEO) and os.path.getsize(COMPRESSED_VIDEO) > 1000:
                orig_mb = os.path.getsize(result_file) / 1e6
                comp_mb = os.path.getsize(COMPRESSED_VIDEO) / 1e6
                shutil.copy(COMPRESSED_VIDEO, final_dest)
                print(f"--- SUCCESS! Saved to Drive: {final_dest} ---")
                print(f"    Original: {orig_mb:.2f} MB → Compressed: {comp_mb:.2f} MB ({100*(1-comp_mb/orig_mb):.1f}% reduction)")
            else:
                print("WARNING: Compression failed, saving uncompressed.")
                shutil.copy(result_file, final_dest)
                print(f"--- Saved uncompressed to: {final_dest} ---")
    else:
        print("CRITICAL: Still 0 frames. Check MODEL_PATH.")
else:
    print("ERROR: Video conversion failed.")

--- Downloading via yt-dlp (section *5.0-170.0) ---
[youtube] Extracting URL: https://www.youtube.com/watch?v=oCgvl2cHzek
[youtube] oCgvl2cHzek: Downloading webpage
[youtube] oCgvl2cHzek: Downloading android vr player API JSON
[info] oCgvl2cHzek: Downloading 1 format(s): 398+140
[info] oCgvl2cHzek: Downloading 1 time ranges: 5.0-170.0
[download] Destination: /content/raw_video.mp4
[libdav1d @ 0x5bd1f67d7900] libdav1d 0.9.2
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from 'https://rr4---sn-npoe7nek.googlevideo.com/videoplayback?expire=1776809636&ei=RKLnabCPHevO4t4PmfzO8Qk&ip=34.142.138.21&id=o-ANccGyUH725cbN_KvVjDiAVSmnNIPFZHBNHi34McQBsZ&itag=398&source=youtube&requiressl=yes&xpc=EgVo2aDSNQ%3D%3D&cps=113&met=1776788036%2C&mh=L6&mm=31%2C29&mn=sn-npoe7nek%2Csn-npoldn76&ms=au%2Crdu&mv=m&mvi=4&pl=17&rms=au%2Cau&initcwndbps=5651250&bui=AUUZDGLaihL0_O5UJSTl7Ca0MxYT4yvl_vjC5Stbv_b4YA7IbgtEGkPL4ZKEiJE7lHB48PqMFwVYXZjD&spc=jlWavSYwqdEiWsrtyN3oZtdf1xlISWvDtOpYxOTCKAgt&vprv=1&svpuc=1&mime=video%2Fmp4&rqh=1

In [ ]:
import os
path = '/content/drive/MyDrive/Sprocket3BS/version_7_frc/detections/'
if os.path.exists(path):
    print("Files found on disk:")
    print(os.listdir(path))
else:
    print("Folder truly does not exist.")

Files found on disk:
[]


In [ ]:
from ultralytics import YOLO
import cv2
from google.colab.patches import cv2_imshow

model = YOLO('/content/drive/MyDrive/Sprocket3BS/version_7_frc/weights/best.pt')

# Test on 1 frame to see if it's working
results = model.predict(source='/content/fixed_video.mp4', frames=1, conf=0.1) # Very low conf
for r in results:
    im_array = r.plot()  # plot a BGR numpy array of predictions
    cv2_imshow(im_array) # This will show the image in Colab

SyntaxError: '[31m[1mframes[0m' is not a valid YOLO argument. Similar arguments are i.e. ['save_frames=False', 'name'].

    Arguments received: ['yolo', '-f', '/root/.local/share/jupyter/runtime/kernel-753d24b2-8901-4d25-9c42-cb789f292ab7.json']. Ultralytics 'yolo' commands use the following syntax:

        yolo TASK MODE ARGS

        Where   TASK (optional) is one of ['obb', 'segment', 'pose', 'detect', 'classify']
                MODE (required) is one of ['predict', 'export', 'benchmark', 'train', 'val', 'track']
                ARGS (optional) are any number of custom 'arg=value' pairs like 'imgsz=320' that override defaults.
                    See all ARGS at https://docs.ultralytics.com/usage/cfg or with 'yolo cfg'

    1. Train a detection model for 10 epochs with an initial learning_rate of 0.01
        yolo train data=coco8.yaml model=yolo26n.pt epochs=10 lr0=0.01

    2. Predict a YouTube video using a pretrained segmentation model at image size 320:
        yolo predi

In [ ]:
import os
import shutil

# Define the source and destination
source_file = '/content/process_test/output/fixed_video.avi'
destination_folder = '/content/drive/MyDrive/Sprocket3BS/version_7_frc/'
destination_path = os.path.join(destination_folder, 'fixed_video.avi')

# Ensure the destination directory exists
os.makedirs(destination_folder, exist_ok=True)

if os.path.exists(source_file):
    print(f"Moving {source_file} to {destination_path}...")
    shutil.move(source_file, destination_path)
    print("Move complete!")
else:
    # If the file isn't there, let's look for it (in case YOLO named the folder differently)
    print(f"Error: {source_file} not found.")
    print("Checking for similar files in process_test...")
    for root, dirs, files in os.walk('/content/process_test'):
        for file in files:
            if file.endswith(".avi"):
                found_path = os.path.join(root, file)
                print(f"Found alternative: {found_path}")

Moving /content/process_test/output/fixed_video.avi to /content/drive/MyDrive/Sprocket3BS/version_7_frc/fixed_video.avi...
Move complete!
